In [ ]:
def load_evaluation_from_fits(fits_path):
    """Load evaluation data from FITS file."""
    with fits.open(fits_path) as hdul:
        # Read metadata
        header = hdul[0].header
        npix = header['NPIX']
        
        # Read catalog
        catalog = Table(hdul['CATALOG'].data)
        
        # Reshape images
        n_samples = len(catalog)
        galaxy_images = catalog['GALAXY_IMAGE'].reshape(n_samples, npix, npix)
        clean_images = catalog['CLEAN_IMAGE'].reshape(n_samples, npix, npix)
        
        if 'PSF_IMAGE' in catalog.colnames:
            psf_images = catalog['PSF_IMAGE'].reshape(n_samples, npix, npix)
        else:
            psf_images = None
        
        # Extract predictions and true values
        true_labels = np.column_stack([
            catalog['TRUE_G1'], catalog['TRUE_G2'],
            catalog['TRUE_SIGMA'], catalog['TRUE_FLUX']
        ])
        nn_preds = np.column_stack([catalog['NN_G1'], catalog['NN_G2']])
        
        ngmix_preds = None
        if 'NGMIX_G1' in catalog.colnames:
            ngmix_preds = np.column_stack([catalog['NGMIX_G1'], catalog['NGMIX_G2']])
        
        # Read response matrices
        response = Table(hdul['RESPONSE'].data)
        R_nn = np.array([[response['R_NN_11'][0], response['R_NN_12'][0]],
                         [response['R_NN_21'][0], response['R_NN_22'][0]]])
        
        return {
            'galaxy_images': galaxy_images,
            'psf_images': psf_images,
            'clean_images': clean_images,
            'true_labels': true_labels,
            'nn_predictions': nn_preds,
            'ngmix_predictions': ngmix_preds,
            'snr': np.array(catalog['SNR']),
            'R_nn': R_nn,
            'catalog': catalog,
            'header': header
        }